# 🐍 Master Guide: Functional Tools (`itertools` & `functools`)
> **Single-Source Revision Guide**: Covers high-performance iterator building blocks (`chain`, `islice`, `groupby`, `product`) and function utilities (`lru_cache`, `singledispatch`).
>
> *Run each cell interactively with `Shift + Enter`.*

---

## 📌 1. Topic Overview
`itertools` and `functools` provide C-implemented primitives for memory-efficient iteration, combinatorics, memoization, and function transformations.

---
## 🧠 2. CPython Architecture: The Power of C-Level Iterators
Functions in `itertools` run compiled in C, consuming $O(1)$ memory and running significantly faster than pure-Python loops.

- **`lru_cache`**: Implemented using a circular doubly-linked list combined with a dictionary for $O(1)$ hit/eviction performance.
- **`itertools.groupby` Requirement**: Input MUST be sorted by the grouping key before calling `groupby()`! CPython's `groupby()` only merges contiguous identical keys.

---
## 🟢 3. Part 1: Beginner Fundamentals

### Step 1: Core `itertools` Utilities (`chain`, `islice`, `cycle`, `count`)

In [ ]:
import itertools

# 1. chain: seamlessly combines multiple iterables without copying
combined = list(itertools.chain([1, 2], [3, 4], [5, 6]))
print("chain output :", combined)

# 2. islice: slices any iterator lazily without materializing a list
sliced = list(itertools.islice(range(100), 10, 15))
print("islice output:", sliced)

# 3. product: Cartesian product (replaces nested loops)
suits = ["Hearts", "Spades"]
ranks = ["A", "K"]
cards = list(itertools.product(ranks, suits))
print("Cartesian product:", cards)

---
## 🟡 4. Part 2: Intermediate Features

### Memoization with `@functools.lru_cache`
Caches return values of expensive or recursive function calls.

In [ ]:
from functools import lru_cache
import time

@lru_cache(maxsize=128)
def fibonacci(n):
    if n < 2:
        return n
    return fibonacci(n - 1) + fibonacci(n - 2)

t0 = time.perf_counter()
result = fibonacci(35)  # Instantaneous with memoization!
t1 = time.perf_counter()

print(f"fibonacci(35) = {result} computed in {(t1 - t0)*1000:.3f} ms")
print("Cache stats:", fibonacci.cache_info())

---
## 🔴 5. Part 3: Tricky Parts, Gotchas & Interview Traps

### ⚠️ Gotcha 1: `itertools.groupby` Unsorted Data Bug
`groupby()` groups only **consecutive** elements! If input is unsorted, duplicate groups are generated.

In [ ]:
data = ["apple", "banana", "apricot", "blueberry", "avocado"]

# Unsorted input bug:
flawed_groups = {k: list(g) for k, g in itertools.groupby(data, key=lambda w: w[0])}
print("Unsorted groupby (Lost earlier 'a' items!):", flawed_groups)

# Correct: Sort data by key first!
sorted_data = sorted(data, key=lambda w: w[0])
correct_groups = {k: list(g) for k, g in itertools.groupby(sorted_data, key=lambda w: w[0])}
print("\nSorted groupby (Correct!):", correct_groups)

---
## 📋 6. Part 4: Quick Revision Cheat Sheet

```
+------------------------------------+---------------------------------------+
| Function                           | Description                           |
+------------------------------------+---------------------------------------+
| itertools.chain(*iterables)        | Merges iterables without copying      |
| itertools.islice(it, start, stop)  | Memory-efficient lazy slicing         |
| itertools.product(*iterables)      | Cartesian product (nested loop replace|
| itertools.groupby(seq, key)        | Groups CONSECUTIVE items (Sort first!)|
| functools.lru_cache(maxsize)       | High-speed O(1) return-value cache    |
| functools.singledispatch           | Method overloading based on arg type  |
+------------------------------------+---------------------------------------+
```

---
## 🎓 Practical Hands-On & Real-World Examples

### 1. Random Security Generators & Captchas (`SampleEx1.py` - `SampleEx3.py`)
Using `random.sample()` to generate secure Captchas, OTPs, and Indian Vehicle Registration Numbers.

In [ ]:
# Practical Implementation: SampleEx1.py - SampleEx3.py
import random
import string

# 1. 6-Character Alphanumeric Captcha Generator (SampleEx2.py)
characters = string.ascii_letters + string.digits + "!@#$%"
captcha_tokens = ["".join(random.sample(characters, k=6)) for _ in range(4)]
print("Generated Captcha Tokens:")
for c in captcha_tokens:
    print(f"  Captcha: {c}")

# 2. Number Plate Generator Simulation (SampleEx3.py)
states = ["TS", "AP", "KA", "MH", "DL"]
districts = ["07", "08", "09", "10", "39"]
series = ["HF", "HL", "AB", "XY"]

print("\nSimulated Vehicle Registration Plates:")
for _ in range(3):
    st = random.choice(states)
    dist = random.choice(districts)
    s_code = random.choice(series)
    num = f"{random.randint(1000, 9999)}"
    print(f"  Vehicle Plate: {st}{dist} {s_code} {num}")

### 2. Regular Expressions Drills (`REG EXPR` - `ExtractMailsEx1.py`, `ExtractNamesEx1.py`)
Validating and extracting emails, mobile numbers, and student records using `re` patterns.

In [ ]:
# Regular Expressions Drills
import re

classroom_log = """
Contact directory:
Rossum (Python Lead) - Email: guido@python.org, Phone: 9845123456
Travis (NumPy Creator) - Email: travis@numpy.org, Phone: 9123456789
Kinney (Pandas Author) - Email: kinney@pandas.py, Phone: 8877665544
Invalid Entry - Email: bad-email@@com, Phone: 1234
"""

# 1. Extract all valid Email addresses
email_pattern = r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}'
extracted_emails = re.findall(email_pattern, classroom_log)
print("Extracted Valid Emails:", extracted_emails)

# 2. Extract valid 10-digit Indian Mobile Numbers (starting with 6, 7, 8, or 9)
phone_pattern = r'\b[6-9]\d{9}\b'
extracted_phones = re.findall(phone_pattern, classroom_log)
print("Extracted Valid Mobile Numbers:", extracted_phones)

# 3. Extract Name & Role pairs
name_role_pattern = r'([A-Z][a-z]+) \((.*?)\)'
extracted_profiles = re.findall(name_role_pattern, classroom_log)
print("\nExtracted Developer Profiles:")
for name, role in extracted_profiles:
    print(f"  {name:<10} -> {role}")